# Настройка окружения для обучения LLM с ClearML

Этот ноутбук поможет вам:
- Настроить **ClearML** для отслеживания экспериментов
- Установить зависимости из репозитория
- Клонировать проект
- Авторизоваться в **Hugging Face Hub**
- Запустить обучение Qwen2.5-0.5B с разными оптимизаторами

> **Совет**: Убедитесь, что в Colab выбран **GPU** (Среда выполнения → Сменить тип среды выполнения → T4 GPU / V100).

# Монтируем Google Drive для сохранения результатов

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Установка всех зависимостей из requirements.txt репозитория
# Используем %pip для корректной работы в Colab
# Предварительно установим git, если его нет (обычно уже есть)


# Клонируем репозиторий
!git clone https://github.com/Se1ecta/huawei-research.git /content/huawei-research

# Устанавливаем зависимости из requirements.txt
!pip install -r /content/huawei-research/requirements.txt

## 🔑 Настройка ClearML

Вам понадобятся **API access key** и **secret key** из вашего аккаунта ClearML (можно получить на [app.clear.ml](https://app.clear.ml) → Settings → Workspace → Create new credentials).

**Вариант A (рекомендуемый)**: Используйте Colab Secrets для безопасного хранения ключей.
1. Откройте `🔑 Секреты` (панель слева в Colab)
2. Добавьте секреты: `CLEARML_API_ACCESS_KEY`, `CLEARML_API_SECRET_KEY`
3. Включите "Разрешить доступ через ноутбук"

**Вариант B**: Вставьте ключи прямо в код (не публикуйте ноутбук с ключами в открытом доступе!).


In [ ]:
import os
from google.colab import userdata

# Пытаемся получить ключи из Colab Secrets
try:
    access_key = userdata.get('CLEARML_API_ACCESS_KEY')
    secret_key = userdata.get('CLEARML_API_SECRET_KEY')
    print("✅ Ключи ClearML загружены из Colab Secrets")
except Exception as e:
    print("⚠️ Не найдены секреты в Colab. Введите ключи вручную (небезопасно для шаринга).")
    access_key = input("Введите CLEARML_API_ACCESS_KEY: ")
    secret_key = input("Введите CLEARML_API_SECRET_KEY: ")

# Устанавливаем переменные окружения для ClearML
os.environ['CLEARML_WEB_HOST'] = 'https://app.clear.ml/'
os.environ['CLEARML_API_HOST'] = 'https://api.clear.ml'
os.environ['CLEARML_FILES_HOST'] = 'https://files.clear.ml'
os.environ['CLEARML_API_ACCESS_KEY'] = access_key
os.environ['CLEARML_API_SECRET_KEY'] = secret_key

print("✅ ClearML настроен. При запуске обучения эксперимент автоматически появится в веб-интерфейсе.")

## Авторизация в Hugging Face Hub

## 🔑 Логин в Hugging Face

Чтобы пушить модели на Hub (опция `--push_to_hub=True`), нужен токен доступа. Получить его можно на [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) (тип `write`).

In [ ]:
from huggingface_hub import notebook_login

notebook_login()
# Появится виджет для ввода токена. Вставьте токен и нажмите "Login".

## Клонируем репозиторий

In [ ]:
!git clone https://github.com/Se1ecta/huawei-research.git

## Авторизация в hugginface

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## Проверка GPU и дополнительных параметров

In [ ]:
import torch

# Проверяем, доступен ли GPU
if torch.cuda.is_available():
    print(f"✅ GPU найден: {torch.cuda.get_device_name(0)}")
    print(f"   Количество памяти: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ GPU не обнаружен. Обучение будет очень медленным. Перезапустите среду выполнения и выберите GPU.")

# Проверяем, что репозиторий склонирован
!ls -la /content/huawei-research/src/

## Запуск обучения (пример с AdamW)

## 🧪 Запуск эксперимента

Команда ниже запускает обучение модели **Qwen/Qwen2.5-0.5B** на 1 эпоху с оптимизатором AdamW.
Все метрики и логи автоматически отправятся в ClearML.

In [ ]:
# Переходим в рабочую директорию (опционально)
import os
os.chdir('/content/huawei-research')

# Можно изменить параметры: batch_size, learning_rate, оптимизатор и т.д.
!python src/train.py \
  --model_name Qwen/Qwen2.5-0.5B \
  --optimizer adamw \
  --per_device_train_batch_size 4 \
  --gradient_accumulation_steps 8 \
  --learning_rate 5e-5 \
  --seq_length 512 \
  --num_train_epochs 1 \
  --lr_scheduler_type cosine \
  --warmup_ratio 0.01 \
  --weight_decay 0.05 \
  --logging_steps 10 \
  --push_to_hub True \
  --report_to clearml \
  --seed 42 \
  --output_dir ./Qwen2.5-0.5B_adamw